#**Atividade para Casa**


---
**Objetivo:** Aplicar GridSearch e Cross-Validation em um modelo de Titanic.

---
##**Instruções:**
1. Baixar o notebook exemplo do repositório.
2. Executar o código completo.
3. Testar as mudanças sugeridas no slide 15.
4. Responder no notebook:


*   Quais foram os melhores hiperparâmetros encontrados?
*   O modelo otimizado teve uma acurácia melhor que o modelo padrão?
*   O que aconteceu quando você mudou o número de folds no Cross-Validation?
*   Você acha que o GridSearch compensa o tempo de processamento? Por quê?

5. Subir o notebook respondido na pasta da Semana 13 do repositório.







##**1. Importar Bibliotecas**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

##**2. Carregar os Dados**

In [ ]:
# Carregar dataset Titanic
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

# Visualizar primeiras linhas
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


##**3. Análise inicial**

In [ ]:
df.info()

df.describe()

df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


##**4. Seleção das Variáveis**

In [ ]:
X = df[['Pclass',
           'Sex',
           'Age',
           'SibSp',
           'Parch',
           'Fare',
           'Embarked']]

y = df['Survived']

##**5. Separação treino e teste**

In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

##**6. Pré-processamento**

###**Variáveis numéricas**

In [ ]:
atributos_numericos = [
    'Age',
    'Fare',
    'SibSp',
    'Parch',
    'Pclass'
]

###**Variáveis categóricas**

In [ ]:
atributos_categoricos = [
    'Sex',
    'Embarked'
]

###**Pipeline numérico**

In [ ]:
transformador_numerico = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

###**Pipeline categórico**

In [ ]:
transformador_categorico = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]
)

###**Column Transformer**

In [ ]:
preprocessador = ColumnTransformer(
    transformers=[
        ('num',
         transformador_numerico,
         atributos_numericos),

        ('cat',
         transformador_categorico,
         atributos_categoricos)
    ]
)

##**7. Modelo Base**

In [ ]:
pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessador),
        ('classifier', LogisticRegression())
    ]
)

##**8. Treinamento**

In [ ]:
pipeline.fit(X_treino, y_treino)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare', 'SibSp',
                                                   'Parch', 'Pclass']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Sex', 'Embarked'])])),
                ('classifier', LogisticRegression())])

##**9. Avaliação do Modelo Base**

In [ ]:
y_pred = pipeline.predict(X_teste)

acuracia_base = accuracy_score(
    y_teste,
    y_pred
)

print("Acurácia do modelo padrão:")
print(acuracia_base)

Acurácia do modelo padrão:
0.8100558659217877


##**10. GridSearchCV**

In [ ]:
param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10, 100],
    'classifier__solver': [
        'liblinear',
        'lbfgs'
    ]
}

In [ ]:
from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

In [ ]:
grid_search.fit(
    X_treino,
    y_treino
)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['Age',
                                                                          'Fare',
                                                                          'SibSp',
                                                                          'Parch',
                                                                          'Pclass']),
                                                                        ('cat',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('onehot',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['Sex',
                                                                          'Embarked'])])),
                                       ('classifier', LogisticRegression())]),
             n_jobs=-1,
             param_grid={'classifier__C': [0.01, 0.1, 1, 10, 100],
                         'classifier__solver': ['liblinear', 'lbfgs']},
             scoring='accuracy')

##**11. Melhores hiperparâmetros**

In [ ]:
print("Melhores hiperparâmetros:")

print(grid_search.best_params_)

Melhores hiperparâmetros:
{'classifier__C': 0.1, 'classifier__solver': 'liblinear'}


##**12. Melhor score do Cross Validation**

In [ ]:
print(
    "Melhor score médio:",
    grid_search.best_score_
)

Melhor score médio: 0.7963065103910174


##**13. Avaliação do modelo otimizado**

In [ ]:
melhor_modelo = grid_search.best_estimator_

y_pred_otimizado = melhor_modelo.predict(X_teste)

acuracia_otimizada = accuracy_score(
    y_teste,
    y_pred_otimizado
)

print(
    "Acurácia do modelo otimizado:",
    acuracia_otimizada
)

Acurácia do modelo otimizado: 0.7988826815642458


##**14. Comparação**

In [ ]:
print(
    f"Modelo padrão: {acuracia_base:.4f}"
)

print(
    f"Modelo otimizado: {acuracia_otimizada:.4f}"
)

Modelo padrão: 0.8101
Modelo otimizado: 0.7989


##**15. Testando diferentes números de folds**

In [ ]:
from sklearn.model_selection import cross_val_score

for cv in [3, 5, 10]:

    scores = cross_val_score(
        melhor_modelo,
        X,
        y,
        cv=cv,
        scoring='accuracy'
    )

    print(f"\nCV = {cv}")

    print(
        f"Média: {scores.mean():.4f}"
    )

    print(
        f"Desvio padrão: {scores.std():.4f}"
    )


CV = 3
Média: 0.7912
Desvio padrão: 0.0027

CV = 5
Média: 0.7913
Desvio padrão: 0.0221

CV = 10
Média: 0.8002
Desvio padrão: 0.0217


##**16. Respostas da atividade**

###**1. Quais foram os melhores hiperparâmetros encontrados?**

Os melhores hiperparâmetros encontrados pelo GridSearchCV foram:

- classifier__C = 0.1
- classifier__solver = 'liblinear'

Esses parâmetros apresentaram o melhor desempenho médio durante o processo de validação cruzada, alcançando um score médio de aproximadamente 0.7963.

###**2. O modelo otimizado teve uma acurácia melhor que o modelo padrão?**

Não.

O modelo padrão obteve uma acurácia de 0.8101 (81,01%), enquanto o modelo otimizado obteve uma acurácia de 0.7989 (79,89%).

Embora o GridSearchCV tenha encontrado a combinação de hiperparâmetros que apresentou o melhor desempenho médio durante a validação cruzada, essa configuração não resultou em uma melhora da acurácia no conjunto de teste. Isso mostra que um melhor desempenho na validação cruzada nem sempre garante um melhor resultado em dados não vistos.

###**3. O que aconteceu quando você mudou o número de folds no Cross-Validation?**

Ao alterar o número de folds da validação cruzada, os resultados sofreram pequenas variações.

- Com 3 folds, a acurácia média foi 0.7912.
- Com 5 folds, a acurácia média foi 0.7913.
- Com 10 folds, a acurácia média foi 0.8002.

Observou-se que o uso de 10 folds produziu a maior acurácia média. Além disso, aumentar o número de folds faz com que mais dados sejam utilizados para treinamento em cada iteração, o que pode gerar estimativas mais confiáveis do desempenho do modelo. Por outro lado, o tempo de processamento também aumenta, pois o modelo precisa ser treinado mais vezes.

###**4. Você acha que o GridSearch compensa o tempo de processamento? Por quê?**

Sim, acredito que o GridSearchCV compensa o tempo de processamento, principalmente em problemas nos quais a maximização do desempenho do modelo é importante.

Apesar de nesse experimento o modelo otimizado não ter superado o modelo padrão no conjunto de teste, o GridSearchCV permitiu testar automaticamente inúmeras combinações de hiperparâmetros e identificar aquela que apresentou o melhor desempenho médio.

Em conjuntos de dados maiores ou em aplicações mais complexas, essa busca sistemática pode resultar em ganhos significativos de desempenho, o que justifica o custo computacional adicional.